In [10]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import pickle as pkl    
from matplotlib import pyplot as plt
from utils_tab import dataframe_to_latex_table
import statsmodels.formula.api as smf


### Setup dati
Importo i dati ed elimino i due soggetti outlier, poi considero solo i blocchi dopo il primo

In [11]:
# with open('results_12_2025.pkl','rb') as f:
#     df = pkl.load(f)

# # Elimino Nisnig che non ha capito la regola
# df = df[df['nickname'] != 'Nisnig']

# # DANMIL è un outlier nei plot
# df = df[df['nickname'] != 'Danmil']  

# # quando analizzo i rule change voglio levare il primo blocco
# # df = df[df['block_number'] > 0]

# # quando analizzo AQ come median split
# df['AQ_score_median_split'] = (df['AQ_score'] >= 21)


In [12]:
import pandas as pd

# Carico il file Excel invece del pickle
df = pd.read_excel('allresults_AQ_IQ.xlsx')

# Elimino Nisnig che non ha capito la regola
df = df[df['nickname'] != 'Nisnig']

# DANMIL è un outlier nei plot
df = df[df['nickname'] != 'Danmil']  

# quando analizzo i rule change voglio levare il primo blocco
# df = df[df['block_number'] > 0]

# quando analizzo AQ come median split
df['AQ_score_median_split'] = (df['AQ_score'] >= 21)

In [13]:
import pandas as pd

# Define your variables of interest
vars_of_interest = ['good_boxes_total', 'boxes_total']

# Define the base columns you want in your final dataframes (including AQ_score if it's already in df)
columns_to_keep = ['nickname','rich_NS', 'block_number', 'trial_in_block_number'] + vars_of_interest 
if 'AQ_score' in df.columns:
    columns_to_keep.append('AQ_score')

# 1. Sort values to ensure perfectly chronological order for our 'shift' logic
df_sorted = df.sort_values(by=['nickname', 'block_number', 'trial_in_block_number']).copy()

# 2. Create temporary shifted columns to look at the "previous" trial within each block
# This entirely replaces the need to do index + 1 or index - 1 inside a loop
df_sorted['prev_is_violation'] = df_sorted.groupby(['nickname', 'block_number'])['is_violation'].shift(1)
df_sorted['prev_trial_num'] = df_sorted.groupby(['nickname', 'block_number'])['trial_in_block_number'].shift(1)


# --- RULE CHANGE DATAFRAMES ---

# Rule change, same: first trial of each block
# (Note: Removed the is_violation == False check from your original code to strictly match your new rules. 
# Add & (df_sorted['is_violation'] == False) if that original condition still applies.)
df_rc_same = df_sorted[
    (df_sorted['trial_in_block_number'] == 0) &
    (df_sorted['is_violation'] == False) &
    (df_sorted['block_number'] > 0)
][columns_to_keep].copy()
df_rc_same['trial_type'] = 'rule_change_same'

# Rule change, next: following each rule change (prev trial was 0), not a violation trial
df_rc_next = df_sorted[
    (df_sorted['prev_trial_num'] == 0) & 
    (df_sorted['is_violation'] == False) &
    (df_sorted['block_number'] > 0)
][columns_to_keep].copy()
df_rc_next['trial_type'] = 'rule_change_next'


# --- VIOLATION DATAFRAMES ---

# Violation, same: any violation which is not the first trial of the block
df_viol_same = df_sorted[
    (df_sorted['is_violation'] == True) & 
    (df_sorted['trial_in_block_number'] != 0)
][columns_to_keep].copy()
df_viol_same['trial_type'] = 'violation_same'

# Violation, next: every trial after a violation which is not a violation itself
df_viol_next = df_sorted[
    (df_sorted['prev_is_violation'] == True) & 
    (df_sorted['is_violation'] == False)
][columns_to_keep].copy()
df_viol_next['trial_type'] = 'violation_next'


# --- NORMAL DATAFRAME (Keeping this just in case you still need it) ---

# Normal: not a violation, not the first trial, and previous trial was not a violation
df_normal = df_sorted[
    (df_sorted['is_violation'] == False) & 
    (df_sorted['trial_in_block_number'] >= 1) &
    (df_sorted['prev_is_violation'] == False)
][columns_to_keep].copy()
df_normal['trial_type'] = 'normal'

# Optional: Clean up the temporary shift columns from the main dataframe if you plan to reuse it
df_sorted.drop(columns=['prev_is_violation', 'prev_trial_num'], inplace=True)


In [14]:
import pandas as pd

# Dizionario con i 4 dataframe da salvare
dfs_to_save = {
    "df_rc_same": df_rc_same,
    "df_rc_next": df_rc_next,
    "df_viol_same": df_viol_same,
    "df_viol_next": df_viol_next
}

# Salvataggio automatico in 4 file Excel separati
for name, dataframe in dfs_to_save.items():
    dataframe.to_excel(f"{name}.xlsx", index=False)

print("I 4 file Excel sono stati salvati correttamente.")

I 4 file Excel sono stati salvati correttamente.


In [15]:
import pandas as pd

# Carica il file
icar = pd.read_excel("AQ&ICAR16_scores.xlsx")

# Tieni solo i soggetti con un punteggio ICAR valido
icar = icar[(icar['ICAR'].notna()) & (icar['ICAR'] != 0)]

# Conta le righe
print(len(icar))

52


In [16]:
import pandas as pd

# Carica il file con i punteggi ICAR
icar = pd.read_excel("AQ&ICAR16_scores.xlsx")

# Tieni solo i soggetti con ICAR valido
icar = (
    icar[(icar["ICAR"].notna()) & (icar["ICAR"] != 0)]
    [["nickname", "ICAR"]]
    .drop_duplicates(subset="nickname")
)

# I file da aggiornare
files = [
    "df_rc_same.xlsx",
    "df_rc_next.xlsx",
    "df_viol_same.xlsx",
    "df_viol_next.xlsx"
]

for file in files:
    # Leggi il file
    df = pd.read_excel(file)

    # Aggiungi la colonna ICAR
    df = df.merge(icar, on="nickname", how="left")

    # Salva sovrascrivendo il file originale
    df.to_excel(file, index=False)

    # Stampa quanti soggetti hanno ricevuto il punteggio ICAR
    print(f"{file}: {df['ICAR'].notna().sum()} righe con ICAR assegnato")

print("Aggiornamento completato.")

df_rc_same.xlsx: 201 righe con ICAR assegnato
df_rc_next.xlsx: 207 righe con ICAR assegnato
df_viol_same.xlsx: 547 righe con ICAR assegnato
df_viol_next.xlsx: 487 righe con ICAR assegnato
Aggiornamento completato.
